# 🏆 Online News Popularity — Champion Solution (Targeting >0.365 Macro F1)
### Kaggle ML Challenge · Multi-Class Classification (A, B, C, D, E) · Metric: Unweighted Macro F1

---

## 🔬 Forensic Post-Mortem: Why Previous Notebooks Capped at ~0.3478
1. **Missing Data Asymmetry (42% NaNs in Train vs 0% in Test)**:
   - In `train.csv`, ~42% of rows contain missing values across various numerical features. In `test.csv`, there are **zero missing values**.
   - Naively feeding NaNs directly into gradient boosting trees causes tree splits to rely on missingness defaults. At test time, zero missing values exist, resulting in train-test distribution shift.
   - Worse: 25+ engineered interaction terms (like ratios and products) propagated NaNs, destroying valuable signal in over 40% of records.
2. **Harmful Adversarial Feature Dropping**:
   - Previous pipelines used adversarial validation with AUC > 0.60 to unconditionally drop top features.
   - In online publishing, temporal and keyword statistics (`timedelta`, `kw_avg_avg`, `kw_max_avg`) naturally trend over time, triggering AUC > 0.60. Dropping these threw away the single strongest predictors of article virality.
3. **Greedy / Flawed Metric Optimization**:
   - Greedy coordinate search on uncalibrated probabilities over a small subset of rows easily overfits or gets stuck in sub-optimal local minima.
4. **Ignoring Target Ordinal Structure**:
   - Popularity categories A through E follow an inherent ordinal spectrum (shares continuum). Treating them purely as unordered 5-class nominal labels discards ordering information between adjacent classes.

---

## 🚀 The 5-Pillar Winning Strategy to Beat 0.365
1. **Pillar 1: Group-Conditioned Imputation & Domain Feature Engineering**:
   - Impute missing values in train using channel-conditioned medians so the training feature distribution matches test's clean distribution.
   - Engineer 45+ domain-specific interaction features: keyword power indices, virality score, content complexity, LDA topic purity/entropy, sentiment contrast, and channel-normalized relative stats.
2. **Pillar 2: Frank & Hall Ordinal Binary Decomposition**:
   - Decomposes the 5-class problem into 4 cumulative binary classifiers ($y \ge B, y \ge C, y \ge D, y \ge E$), reconstructing mathematically guaranteed ordinal probabilities that sharpen boundary separation (especially rare classes A and E).
3. **Pillar 3: Diverse Multi-Model Architecture**:
   - **LightGBM Multiclass** (CPU-optimized fast histogram gradient boosting)
   - **XGBoost Multiclass** (GPU-accelerated hist with depth regularization)
   - **CatBoost Multiclass** (GPU symmetric trees with robust categorical handling)
   - **Ordinal Binary Decomposition Classifiers** (Frank-Hall cumulative LightGBM/XGBoost)
   - **Continuous Ordinal Regressors** (LightGBM on class index ranks)
4. **Pillar 4: Stacking & Dirichlet Meta-Blending**:
   - Log-odds probability transformation stacked with continuous ordinal predictions via cross-validated regularized meta-learners.
5. **Pillar 5: Multi-Restart Nelder-Mead Threshold Optimization for Macro F1**:
   - Systematically tunes class decision multipliers via 5-fold cross-validation with multiple random restarts to directly maximize Macro F1, boosting rare class recall without sacrificing precision.

---
**Kaggle Setup**: Accelerator → **GPU T4 x2** (or P100) → Ensure competition dataset is attached → **Run All**. Output: `/kaggle/working/submission.csv`.

In [99]:
import os, glob, time, warnings, json, gc, itertools
import numpy as np, pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, log_loss, classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor

warnings.filterwarnings('ignore')

# ----------------------------------------------------
# Global Configuration
# ----------------------------------------------------
SEED = 42
N_FOLDS = 5
FAST_MODE = False             # Set True for a quick 2-minute smoke test
USE_GPU = True                # Uses GPU for XGBoost and CatBoost
N_SEEDS = 2                   # Multi-seed iterations for variance reduction
RUN_CATBOOST = True           # Include CatBoost in the ensemble
RUN_ORDINAL_DECOMP = True     # Frank & Hall Ordinal Binary Decomposition
RUN_ORDINAL_REG = True        # Continuous Ordinal Regressor

np.random.seed(SEED)
print(f"Setup complete | SEED={SEED} | N_FOLDS={N_FOLDS} | N_SEEDS={N_SEEDS} | FAST_MODE={FAST_MODE}")

Setup complete | SEED=42 | N_FOLDS=5 | N_SEEDS=2 | FAST_MODE=False


In [100]:
def find_dataset_file(name):
    """Locate file dynamically across Kaggle input directories and local directories."""
    patterns = [
        f'/kaggle/input/**/{name}',
        f'./**/{name}',
        f'../input/**/{name}',
        f'./ml-challenge-cs-club/{name}'
    ]
    for pattern in patterns:
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"Could not find {name}. Please attach the dataset to your Kaggle session.")

train_path = find_dataset_file('train.csv')
test_path  = find_dataset_file('test.csv')

train = pd.read_csv(train_path, na_values=['NA', 'nan', ''])
test  = pd.read_csv(test_path,  na_values=['NA', 'nan', ''])

print(f"Loaded train: {train.shape} | test: {test.shape}")

# Target class mapping
classes = sorted(train['target_popularity'].dropna().unique())
K = len(classes)
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for i, c in enumerate(classes)}

y = train['target_popularity'].map(class_to_idx).values

print("Target Classes:", classes)
class_dist = pd.Series(y).value_counts(normalize=True).sort_index().rename('Train Proportion')
class_dist.index = [idx_to_class[i] for i in class_dist.index]
print("\nClass Distribution:")
print(class_dist.round(4))

Loaded train: (37144, 61) | test: (2500, 60)
Target Classes: ['A', 'B', 'C', 'D', 'E']

Class Distribution:
A    0.0462
B    0.1062
C    0.1946
D    0.3636
E    0.2893
Name: Train Proportion, dtype: float64


In [101]:
def extract_channel_and_weekday(df):
    """Extract discrete channel_id and weekday_id from one-hot columns."""
    d = df.copy()
    ch_cols = [c for c in d.columns if c.startswith('data_channel_is_')]
    if ch_cols:
        ch_vals = d[ch_cols].fillna(0).values
        has_ch = ch_vals.sum(axis=1) > 0
        d['channel_id'] = np.where(has_ch, ch_vals.argmax(axis=1), -1)
    else:
        d['channel_id'] = -1
        
    wd_cols = [c for c in d.columns if c.startswith('weekday_is_')]
    if wd_cols:
        wd_vals = d[wd_cols].fillna(0).values
        d['weekday_id'] = wd_vals.argmax(axis=1)
    else:
        d['weekday_id'] = 0
        
    return d

train_init = extract_channel_and_weekday(train)
test_init  = extract_channel_and_weekday(test)

# Check missing values
na_train = train.drop(columns=['id', 'target_popularity']).isna().sum()
na_cols = na_train[na_train > 0].index.tolist()
print(f"Columns with NAs in train: {len(na_cols)} | Total NAs in test: {test.isna().sum().sum()}")

def impute_missing_data(train_df, test_df, num_cols):
    """
    Impute numeric missing values using group-wise channel medians,
    falling back to global medians. Ensures train matches test's clean distribution.
    """
    tr = train_df.copy()
    te = test_df.copy()
    
    for c in num_cols:
        if c in tr.columns and tr[c].isna().sum() > 0:
            grp_med = tr.groupby('channel_id')[c].transform('median')
            global_med = tr[c].median()
            tr[c] = tr[c].fillna(grp_med).fillna(global_med)
            if c in te.columns and te[c].isna().sum() > 0:
                te_grp_med = te.groupby('channel_id')[c].transform('median')
                te[c] = te[c].fillna(te_grp_med).fillna(global_med)
                
    return tr, te

feat_cols = [c for c in train.columns if c not in ['id', 'target_popularity']]
train_imp, test_imp = impute_missing_data(train_init, test_init, feat_cols)
print(f"After imputation: Train NAs = {train_imp[feat_cols].isna().sum().sum()} | Test NAs = {test_imp[feat_cols].isna().sum().sum()}")

Columns with NAs in train: 15 | Total NAs in test: 0
After imputation: Train NAs = 0 | Test NAs = 0


In [102]:
def engineer_features(df):
    """
    Comprehensive feature engineering for Mashable Online News Popularity:
    1. Keyword power & dispersion
    2. Self-reference virality
    3. Content density & multimedia
    4. LDA topic geometry & purity
    5. Sentiment synergy & extremity
    6. Temporal & channel interactions
    """
    d = df.copy()
    eps = 1e-6
    
    # --- 1. Keywords (Strongest predictive signal) ---
    kw_num = ['kw_avg_avg', 'kw_max_avg', 'kw_min_avg', 'kw_avg_max', 'kw_min_max', 'kw_max_min', 'kw_avg_min']
    for c in kw_num:
        if c in d.columns:
            d[f'log_{c}'] = np.log1p(np.maximum(0, d[c]))
            
    d['kw_spread_avg'] = d['kw_max_avg'] - d['kw_min_avg']
    d['kw_ratio_avg'] = d['kw_avg_avg'] / (d['kw_max_avg'] + 1.0)
    d['kw_min_to_avg'] = d['kw_min_avg'] / (d['kw_avg_avg'] + 1.0)
    d['kw_max_min_ratio'] = d['kw_max_min'] / (d['kw_avg_min'] + 1.0)
    d['kw_power_composite'] = d['log_kw_avg_avg'] * 0.5 + d['log_kw_max_avg'] * 0.3 + d['log_kw_avg_max'] * 0.2
    d['kw_virality_product'] = d['log_kw_avg_avg'] * d['log_kw_max_avg']
    d['kw_avg_x_channel'] = d['kw_avg_avg'] * (d['channel_id'] + 2)
    d['kw_avg_to_time'] = d['kw_avg_avg'] / (d['timedelta'] + 10.0)
    d['kw_high_tier'] = (d['kw_avg_avg'] > 4000).astype(float)
    d['kw_low_tier'] = (d['kw_avg_avg'] < 1400).astype(float)

    # --- 2. Self-Reference Shares ---
    d['has_self_ref'] = (d['num_self_hrefs'] > 0).astype(float)
    d['log_selfref_avg'] = np.log1p(np.maximum(0, d['self_reference_avg_sharess']))
    d['log_selfref_max'] = np.log1p(np.maximum(0, d['self_reference_max_shares']))
    d['log_selfref_min'] = np.log1p(np.maximum(0, d['self_reference_min_shares']))
    d['selfref_spread'] = d['self_reference_max_shares'] - d['self_reference_min_shares']
    d['selfref_ratio'] = d['self_reference_max_shares'] / (d['self_reference_min_shares'] + 1.0)
    d['selfref_to_kw'] = d['log_selfref_avg'] / (d['log_kw_avg_avg'] + 1.0)
    d['selfref_x_links'] = d['log_selfref_avg'] * d['num_self_hrefs']

    # --- 3. Content Structure & Multimedia ---
    d['log_tokens_content'] = np.log1p(np.maximum(0, d['n_tokens_content']))
    d['is_zero_content'] = (d['n_tokens_content'] == 0).astype(float)
    d['links_per_token'] = d['num_hrefs'] / (d['n_tokens_content'] + 1.0)
    d['self_link_ratio'] = d['num_self_hrefs'] / (d['num_hrefs'] + 1.0)
    d['media_total'] = d['num_imgs'] + d['num_videos']
    d['media_per_token'] = d['media_total'] / (d['n_tokens_content'] + 1.0)
    d['imgs_per_token'] = d['num_imgs'] / (d['n_tokens_content'] + 1.0)
    d['has_video'] = (d['num_videos'] > 0).astype(float)
    d['has_img'] = (d['num_imgs'] > 0).astype(float)
    d['title_content_ratio'] = d['n_tokens_title'] / (d['n_tokens_content'] + 1.0)
    d['kw_per_title'] = d['num_keywords'] / (d['n_tokens_title'] + 1.0)
    d['token_complexity'] = d['average_token_length'] * d['n_unique_tokens'].clip(0, 1)

    # --- 4. LDA Topics Geometry ---
    lda_cols = [f'LDA_0{i}' for i in range(5)]
    lda_arr = d[lda_cols].values
    d['lda_max'] = lda_arr.max(axis=1)
    d['lda_argmax'] = lda_arr.argmax(axis=1)
    d['lda_entropy'] = -(d[lda_cols].clip(eps, 1.0) * np.log(d[lda_cols].clip(eps, 1.0))).sum(axis=1)
    sorted_lda = np.sort(lda_arr, axis=1)
    d['lda_second'] = sorted_lda[:, -2]
    d['lda_margin'] = d['lda_max'] - d['lda_second']  # Topic clarity / purity

    # --- 5. Sentiment & Emotion Synergy ---
    d['pos_neg_diff'] = d['global_rate_positive_words'] - d['global_rate_negative_words']
    d['pos_neg_ratio'] = d['global_rate_positive_words'] / (d['global_rate_negative_words'] + 1e-5)
    d['pol_range_pos'] = d['max_positive_polarity'] - d['min_positive_polarity']
    d['pol_range_neg'] = d['max_negative_polarity'] - d['min_negative_polarity']
    d['title_intensity'] = d['title_subjectivity'] * d['abs_title_sentiment_polarity']
    d['subj_x_sent'] = d['global_subjectivity'] * d['global_sentiment_polarity']
    d['sentiment_contrast'] = d['avg_positive_polarity'] - np.abs(d['avg_negative_polarity'])
    d['title_clickbait'] = d['title_subjectivity'] * np.abs(d['title_sentiment_polarity']) / (d['n_tokens_title'] + 1.0)

    # --- 6. Temporal Cycles & Channels ---
    d['timedelta_years'] = d['timedelta'] / 365.25
    d['timedelta_decay'] = np.exp(-d['timedelta'] / 365.25)
    d['weekday_sin'] = np.sin(2 * np.pi * d['weekday_id'] / 7.0)
    d['weekday_cos'] = np.cos(2 * np.pi * d['weekday_id'] / 7.0)
    d['is_tech_or_socmed'] = ((d['channel_id'] == 3) | (d['channel_id'] == 4)).astype(float)
    d['virality_index'] = (d['num_hrefs'] + 2*d['num_self_hrefs'] + 3*d['num_imgs'] + 4*d['num_videos']) / (d['log_tokens_content'] + 1.0)
    
    return d

X_all = engineer_features(train_imp.drop(columns=['id', 'target_popularity']))
Xt_all = engineer_features(test_imp.drop(columns=['id']))[X_all.columns]

# Channel-relative normalizations
group_cols = ['log_kw_avg_avg', 'log_tokens_content', 'global_subjectivity', 'media_total']
for gc in group_cols:
    if gc in X_all.columns:
        means = X_all.groupby('channel_id')[gc].mean()
        X_all[f'{gc}_diff_channel'] = X_all[gc] - X_all['channel_id'].map(means)
        Xt_all[f'{gc}_diff_channel'] = Xt_all[gc] - Xt_all['channel_id'].map(means)

print(f"Final Feature Matrix: Train={X_all.shape}, Test={Xt_all.shape}")

Final Feature Matrix: Train=(37144, 121), Test=(2500, 121)


In [103]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X_all, y))

# Storage for model out-of-fold and test predictions
model_oofs = {}
model_preds = {}

def evaluate_predictions(name, oof_probs, y_true):
    """Evaluate log-loss, Macro F1, and per-class F1."""
    pred_labels = oof_probs.argmax(axis=1)
    loss = log_loss(y_true, oof_probs)
    f1_macro = f1_score(y_true, pred_labels, average='macro')
    per_class = f1_score(y_true, pred_labels, average=None)
    f1_dict = {classes[i]: round(per_class[i], 4) for i in range(K)}
    print(f"[{name}] LogLoss: {loss:.4f} | Macro F1 (argmax): {f1_macro:.4f}")
    print(f"  Per-class F1: {f1_dict}")
    return {'logloss': loss, 'macro_f1': f1_macro, 'per_class': f1_dict}

In [104]:
def train_lgb_multiclass(X_tr, y_tr, X_te, seeds=[SEED]):
    oof_accum = np.zeros((len(X_tr), K))
    pred_accum = np.zeros((len(X_te), K))
    n_est = 200 if FAST_MODE else 2500
    
    for s_idx, s in enumerate(seeds):
        print(f"--- LightGBM Multiclass (Seed {s_idx+1}/{len(seeds)}) ---")
        oof_seed = np.zeros((len(X_tr), K))
        pred_seed = np.zeros((len(X_te), K))
        
        for k, (tr, va) in enumerate(folds):
            m = lgb.LGBMClassifier(
                n_estimators=n_est,
                learning_rate=0.03,
                num_leaves=42,
                min_child_samples=35,
                subsample=0.8,
                subsample_freq=1,
                colsample_bytree=0.65,
                reg_lambda=3.0,
                reg_alpha=0.5,
                objective='multiclass',
                num_class=K,
                random_state=s + k,
                n_jobs=-1,
                verbose=-1
            )
            m.fit(
                X_tr.iloc[tr], y_tr[tr],
                eval_set=[(X_tr.iloc[va], y_tr[va])],
                callbacks=[lgb.early_stopping(80, verbose=False)]
            )
            val_p = m.predict_proba(X_tr.iloc[va])
            oof_seed[va] = val_p
            pred_seed += m.predict_proba(X_te) / N_FOLDS
            
        oof_accum += oof_seed / len(seeds)
        pred_accum += pred_seed / len(seeds)
        
    return oof_accum, pred_accum

lgb_seeds = [SEED + 100 * i for i in range(N_SEEDS)]
t0 = time.time()
lgb_oof, lgb_pred = train_lgb_multiclass(X_all, y, Xt_all, seeds=lgb_seeds)
model_oofs['lgb_multi'] = lgb_oof
model_preds['lgb_multi'] = lgb_pred
evaluate_predictions("LightGBM Multiclass", lgb_oof, y)
print(f"LightGBM completed in {time.time()-t0:.1f}s\n")

--- LightGBM Multiclass (Seed 1/2) ---
--- LightGBM Multiclass (Seed 2/2) ---
[LightGBM Multiclass] LogLoss: 1.2995 | Macro F1 (argmax): 0.2667
  Per-class F1: {'A': np.float64(0.0324), 'B': np.float64(0.0588), 'C': np.float64(0.2101), 'D': np.float64(0.5047), 'E': np.float64(0.5277)}
LightGBM completed in 171.7s



In [ ]:
def train_xgb_multiclass(X_tr, y_tr, X_te, seeds=[SEED]):
    oof_accum = np.zeros((len(X_tr), K))
    pred_accum = np.zeros((len(X_te), K))
    n_est = 200 if FAST_MODE else 2500
    
    xgb_device = 'cpu'
    if USE_GPU:
        try:
            import torch
            if torch.cuda.is_available():
                xgb_device = 'cuda'
        except Exception:
            xgb_device = 'cuda'
            
    for s_idx, s in enumerate(seeds):
        print(f"--- XGBoost Multiclass (Seed {s_idx+1}/{len(seeds)}) [Device={xgb_device}] ---")
        oof_seed = np.zeros((len(X_tr), K))
        pred_seed = np.zeros((len(X_te), K))
        
        for k, (tr, va) in enumerate(folds):
            m = xgb.XGBClassifier(
                n_estimators=n_est,
                learning_rate=0.03,
                max_depth=6,
                min_child_weight=6,
                subsample=0.8,
                colsample_bytree=0.65,
                reg_lambda=4.0,
                reg_alpha=0.5,
                tree_method='hist',
                device=xgb_device,
                objective='multi:softprob',
                num_class=K,
                eval_metric='mlogloss',
                early_stopping_rounds=80,
                random_state=s + k,
                n_jobs=4
            )
            m.fit(
                X_tr.iloc[tr], y_tr[tr],
                eval_set=[(X_tr.iloc[va], y_tr[va])],
                verbose=False
            )
            val_p = m.predict_proba(X_tr.iloc[va])
            oof_seed[va] = val_p
            pred_seed += m.predict_proba(X_te) / N_FOLDS
            
        oof_accum += oof_seed / len(seeds)
        pred_accum += pred_seed / len(seeds)
        
    return oof_accum, pred_accum

xgb_seeds = [SEED + 100 * i for i in range(N_SEEDS)]
t0 = time.time()
xgb_oof, xgb_pred = train_xgb_multiclass(X_all, y, Xt_all, seeds=xgb_seeds)
model_oofs['xgb_multi'] = xgb_oof
model_preds['xgb_multi'] = xgb_pred
evaluate_predictions("XGBoost Multiclass", xgb_oof, y)
print(f"XGBoost completed in {time.time()-t0:.1f}s\n")

--- XGBoost Multiclass (Seed 1/2) [Device=cuda] ---
--- XGBoost Multiclass (Seed 2/2) [Device=cuda] ---


In [ ]:
if RUN_CATBOOST:
    def train_cat_multiclass(X_tr, y_tr, X_te, seeds=[SEED]):
        oof_accum = np.zeros((len(X_tr), K))
        pred_accum = np.zeros((len(X_te), K))
        n_est = 200 if FAST_MODE else 3000
        
        for s_idx, s in enumerate(seeds):
            print(f"--- CatBoost Multiclass (Seed {s_idx+1}/{len(seeds)}) ---")
            oof_seed = np.zeros((len(X_tr), K))
            pred_seed = np.zeros((len(X_te), K))
            
            for k, (tr, va) in enumerate(folds):
                try:
                    m = CatBoostClassifier(
                        iterations=n_est,
                        learning_rate=0.04,
                        depth=6,
                        l2_leaf_reg=5.0,
                        loss_function='MultiClass',
                        eval_metric='MultiClass',
                        task_type='GPU' if USE_GPU else 'CPU',
                        random_seed=s + k,
                        od_type='Iter',
                        od_wait=100,
                        verbose=False
                    )
                    m.fit(X_tr.iloc[tr], y_tr[tr], eval_set=(X_tr.iloc[va], y_tr[va]))
                except Exception as e:
                    print(f"CatBoost GPU fallback to CPU on Fold {k} ({e})")
                    m = CatBoostClassifier(
                        iterations=min(n_est, 1000),
                        learning_rate=0.05,
                        depth=6,
                        l2_leaf_reg=5.0,
                        loss_function='MultiClass',
                        random_seed=s + k,
                        early_stopping_rounds=80,
                        verbose=False
                    )
                    m.fit(X_tr.iloc[tr], y_tr[tr], eval_set=(X_tr.iloc[va], y_tr[va]))
                    
                oof_seed[va] = m.predict_proba(X_tr.iloc[va])
                pred_seed += m.predict_proba(X_te) / N_FOLDS
                
            oof_accum += oof_seed / len(seeds)
            pred_accum += pred_seed / len(seeds)
            
        return oof_accum, pred_accum

    cat_seeds = [SEED + 100 * i for i in range(1 if FAST_MODE else N_SEEDS)]
    t0 = time.time()
    cat_oof, cat_pred = train_cat_multiclass(X_all, y, Xt_all, seeds=cat_seeds)
    model_oofs['cat_multi'] = cat_oof
    model_preds['cat_multi'] = cat_pred
    evaluate_predictions("CatBoost Multiclass", cat_oof, y)
    print(f"CatBoost completed in {time.time()-t0:.1f}s\n")

In [ ]:
if RUN_ORDINAL_DECOMP:
    print("=" * 60)
    print("Training Frank & Hall Ordinal Binary Decomposition")
    print("Decomposing 5 classes into 4 cumulative binary boundary models:")
    print("  Cutoff 0: y >= 1 (B, C, D, E vs A)")
    print("  Cutoff 1: y >= 2 (C, D, E vs A, B)")
    print("  Cutoff 2: y >= 3 (D, E vs A, B, C)")
    print("  Cutoff 3: y >= 4 (E vs A, B, C, D)")
    print("=" * 60)
    
    n_est = 200 if FAST_MODE else 2000
    cum_oofs = []
    cum_preds = []
    
    t0 = time.time()
    for cutoff in range(1, K):
        y_binary = (y >= cutoff).astype(int)
        c_oof = np.zeros(len(X_all))
        c_pred = np.zeros(len(Xt_all))
        
        for k, (tr, va) in enumerate(folds):
            m = lgb.LGBMClassifier(
                n_estimators=n_est,
                learning_rate=0.03,
                num_leaves=36,
                min_child_samples=40,
                subsample=0.8,
                subsample_freq=1,
                colsample_bytree=0.65,
                reg_lambda=3.0,
                objective='binary',
                random_state=SEED + cutoff * 10 + k,
                n_jobs=-1,
                verbose=-1
            )
            m.fit(
                X_all.iloc[tr], y_binary[tr],
                eval_set=[(X_all.iloc[va], y_binary[va])],
                callbacks=[lgb.early_stopping(80, verbose=False)]
            )
            c_oof[va] = m.predict_proba(X_all.iloc[va])[:, 1]
            c_pred += m.predict_proba(Xt_all)[:, 1] / N_FOLDS
            
        cum_oofs.append(c_oof)
        cum_preds.append(c_pred)
        auc = roc_auc_score(y_binary, c_oof)
        print(f"Cutoff {cutoff-1} (y >= {classes[cutoff]}): ROC-AUC = {auc:.4f}")
        
    def reconstruct_ordinal_probs(cum_list, N):
        """Reconstruct 5 class probabilities from 4 cumulative probabilities."""
        Q = np.column_stack(cum_list)  # (N, 4): Q[:, i] = P(y >= i+1)
        P = np.zeros((N, K))
        P[:, 0] = 1.0 - Q[:, 0]
        P[:, 1] = np.maximum(0, Q[:, 0] - Q[:, 1])
        P[:, 2] = np.maximum(0, Q[:, 1] - Q[:, 2])
        P[:, 3] = np.maximum(0, Q[:, 2] - Q[:, 3])
        P[:, 4] = Q[:, 3]
        # Normalize rows to sum to 1
        P = P / P.sum(axis=1, keepdims=True)
        return P

    ord_oof = reconstruct_ordinal_probs(cum_oofs, len(X_all))
    ord_pred = reconstruct_ordinal_probs(cum_preds, len(Xt_all))
    
    model_oofs['frank_hall_ordinal'] = ord_oof
    model_preds['frank_hall_ordinal'] = ord_pred
    evaluate_predictions("Frank & Hall Ordinal", ord_oof, y)
    print(f"Ordinal Decomposition completed in {time.time()-t0:.1f}s\n")

In [ ]:
if RUN_ORDINAL_REG:
    print("Training Continuous Ordinal Regressors on Class Ranks (0, 1, 2, 3, 4)...")
    reg_oof = np.zeros(len(X_all))
    reg_pred = np.zeros(len(Xt_all))
    n_est = 200 if FAST_MODE else 2500
    
    for k, (tr, va) in enumerate(folds):
        m = lgb.LGBMRegressor(
            n_estimators=n_est,
            learning_rate=0.03,
            num_leaves=32,
            min_child_samples=40,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.65,
            reg_lambda=3.0,
            objective='regression',
            random_state=SEED + k,
            n_jobs=-1,
            verbose=-1
        )
        m.fit(
            X_all.iloc[tr], y[tr],
            eval_set=[(X_all.iloc[va], y[va])],
            callbacks=[lgb.early_stopping(80, verbose=False)]
        )
        reg_oof[va] = m.predict(X_all.iloc[va])
        reg_pred += m.predict(Xt_all) / N_FOLDS
        
    corr = np.corrcoef(reg_oof, y)[0, 1]
    print(f"Ordinal Regressor correlation with true label rank: {corr:.4f}")

In [ ]:
print("=" * 60)
print("Evaluating Ensembling Strategies")
print("=" * 60)

available_models = list(model_oofs.keys())
print("Candidate probability models:", available_models)

# 1. Optimal Dirichlet / Softmax Log-Loss Blend
def loss_func(weights):
    w = np.exp(weights) / np.sum(np.exp(weights))
    P = sum(w[i] * model_oofs[m] for i, m in enumerate(available_models))
    P = np.clip(P, 1e-6, 1.0 - 1e-6)
    return log_loss(y, P)

init_w = np.zeros(len(available_models))
res = minimize(loss_func, init_w, method='Nelder-Mead', options={'maxiter': 500})
best_weights = np.exp(res.x) / np.sum(np.exp(res.x))

print("\nOptimal Model Blend Weights:")
for m, w_val in zip(available_models, best_weights):
    print(f"  {m}: {w_val:.4f}")

blend_oof = sum(w * model_oofs[m] for w, m in zip(best_weights, available_models))
blend_pred = sum(w * model_preds[m] for w, m in zip(best_weights, available_models))
evaluate_predictions("Optimal Weighted Blend", blend_oof, y)

# 2. Meta-Stacking with Multinomial Logistic Regression
stack_X_oof_list = [np.log(np.clip(model_oofs[m], 1e-6, 1.0)) for m in available_models]
stack_X_test_list = [np.log(np.clip(model_preds[m], 1e-6, 1.0)) for m in available_models]

if RUN_ORDINAL_REG:
    stack_X_oof_list.append(reg_oof.reshape(-1, 1))
    stack_X_test_list.append(reg_pred.reshape(-1, 1))
    
S_oof = np.hstack(stack_X_oof_list)
S_test = np.hstack(stack_X_test_list)

meta_pipe = make_pipeline(StandardScaler(), LogisticRegression(C=0.2, max_iter=2000, multi_class='multinomial', random_state=SEED))
stack_oof = cross_val_predict(meta_pipe, S_oof, y, cv=StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED+9), method='predict_proba')
meta_pipe.fit(S_oof, y)
stack_pred = meta_pipe.predict_proba(S_test)
evaluate_predictions("Stacking Meta-Learner", stack_oof, y)

# Pick best probability baseline based on Macro F1
if f1_score(y, stack_oof.argmax(1), average='macro') > f1_score(y, blend_oof.argmax(1), average='macro'):
    print("\n>>> Winner for probability generation: Stacking Meta-Learner")
    chosen_oof = stack_oof
    chosen_pred = stack_pred
else:
    print("\n>>> Winner for probability generation: Optimal Weighted Blend")
    chosen_oof = blend_oof
    chosen_pred = blend_pred

In [ ]:
print("=" * 60)
print("🎯 The Secret Weapon: CV-Regularized Macro F1 Multiplier Tuning")
print("Tuning decision multipliers: pred_label = argmax_c (w_c * p_c)")
print("=" * 60)

def macro_f1_loss(w, probs, targets):
    """Negative Macro F1 objective function."""
    scaled = probs * w
    preds = scaled.argmax(axis=1)
    return -f1_score(targets, preds, average='macro', zero_division=0)

def optimize_weights_cv(probs, targets, cv_folds, n_restarts=12):
    """
    Tune multiplier vector using multi-restart Nelder-Mead on out-of-fold data.
    Cross-validation ensures weights generalize and do not overfit.
    """
    fold_weights = []
    fold_f1_raw = []
    fold_f1_tuned = []
    
    for k, (tr, va) in enumerate(cv_folds):
        P_tr, y_tr = probs[tr], targets[tr]
        P_va, y_va = probs[va], targets[va]
        
        raw_score = f1_score(y_va, P_va.argmax(axis=1), average='macro')
        fold_f1_raw.append(raw_score)
        
        best_val_loss = float('inf')
        best_w = np.ones(K)
        
        # Multiple random restarts to avoid local minima
        for restart in range(n_restarts):
            init_w = np.ones(K) if restart == 0 else np.random.uniform(0.6, 1.8, size=K)
            res = minimize(
                macro_f1_loss,
                init_w,
                args=(P_tr, y_tr),
                method='Nelder-Mead',
                options={'maxiter': 300, 'xatol': 1e-3, 'fatol': 1e-4}
            )
            if res.fun < best_val_loss:
                best_val_loss = res.fun
                best_w = res.x
                
        # Normalize weights so mean is 1.0
        best_w = np.maximum(best_w, 0.1)
        best_w = best_w / np.mean(best_w)
        fold_weights.append(best_w)
        
        tuned_score = f1_score(y_va, (P_va * best_w).argmax(axis=1), average='macro')
        fold_f1_tuned.append(tuned_score)
        print(f"Fold {k+1}: Raw Macro F1 = {raw_score:.4f} --> Tuned Macro F1 = {tuned_score:.4f} (+{tuned_score-raw_score:.4f})")
        
    print(f"\nMean Out-of-Fold Macro F1: Raw = {np.mean(fold_f1_raw):.4f} --> Tuned = {np.mean(fold_f1_tuned):.4f} (+{np.mean(fold_f1_tuned)-np.mean(fold_f1_raw):.4f})")
    
    # Average optimal weights across folds for the final submission
    ensemble_weights = np.mean(fold_weights, axis=0)
    ensemble_weights = ensemble_weights / np.mean(ensemble_weights)
    return ensemble_weights

optimal_weights = optimize_weights_cv(chosen_oof, y, folds, n_restarts=12)

print("\nFinal Ensemble Multiplier Vector (per class):")
for i, c in enumerate(classes):
    print(f"  Class {c}: {optimal_weights[i]:.4f}")

# Full Out-of-Fold Final Evaluation
final_oof_preds = (chosen_oof * optimal_weights).argmax(axis=1)
final_macro_f1 = f1_score(y, final_oof_preds, average='macro')
print(f"\n🔥 OVERALL TUNED OUT-OF-FOLD MACRO F1: {final_macro_f1:.4f} 🔥")

print("\nClassification Report (Out-of-Fold):")
print(classification_report(y, final_oof_preds, target_names=classes, digits=4))

In [ ]:
print("=" * 60)
print("Writing Final Kaggle Submission")
print("=" * 60)

# Compute final test predictions using tuned weights
final_test_labels_idx = (chosen_pred * optimal_weights).argmax(axis=1)
final_test_classes = [classes[i] for i in final_test_labels_idx]

submission = pd.DataFrame({
    'id': test['id'],
    'target_popularity': final_test_classes
})

submission.to_csv('submission.csv', index=False)
print("Saved submission to submission.csv")

# --- Rigorous Integrity & Quality Checks ---
assert len(submission) == len(test), f"Length mismatch: {len(submission)} vs {len(test)}"
assert submission['id'].nunique() == len(test), "Duplicate IDs detected!"
assert submission['target_popularity'].isna().sum() == 0, "Missing values detected in predictions!"
assert set(submission['target_popularity'].unique()).issubset(set(classes)), f"Invalid classes: {submission['target_popularity'].unique()}"

print("\nIntegrity Checks Passed:")
print(f"  Total Rows: {len(submission)} (matches test.csv)")
print(f"  Unique IDs: {submission['id'].nunique()}")
print(f"  Columns: {list(submission.columns)}")

# Compare predicted class distribution against training ground truth
print("\nPredicted Class Distribution vs Train Distribution:")
comp_df = pd.concat([
    submission['target_popularity'].value_counts(normalize=True).sort_index().rename('Test Pred %'),
    pd.Series(y).map(idx_to_class).value_counts(normalize=True).sort_index().rename('Train True %')
], axis=1) * 100
print(comp_df.round(2))

print("\nFirst 10 Predictions:")
print(submission.head(10))
print("\n🎉 Submission ready to upload to Kaggle! Good luck achieving 1st place! 🚀")